# Preparar y estructurar los datos de Instacart con pandas

**Evidencia:** GA1-220501094-01-AA1-EV06 – Limpieza, análisis y estructuración de datos con Pandas (Instacart)

**Aprendiz:** Maicol Esneider Posada Callejas · **Ficha:** 3223873 · **Instructor:** Carlos Navia

**Pregunta orientadora:** ¿Cómo preparar y estructurar los datos de Instacart para obtener información confiable sobre los hábitos de compra?

Este cuaderno desarrolla las lecciones 2, 3 y 4 del material *Preparar y estructurar datos con Python*. Cada lección tiene tres partes: los ejemplos de la explicación, el laboratorio guiado y el reto de 7 ejercicios. La exploración inicial (Lección 1) corresponde a `data_quality.ipynb`, de la actividad anterior.

**Contenido**

0. Preparación: carga de los datos crudos
1. Lección 2: Limpieza y estandarización de datos crudos
2. Lección 3: Filtrado, ordenamiento y agrupación de datos
3. Lección 4: Combinación y estructuración de múltiples datasets

## 0. Preparación: carga de los datos crudos

Los cinco CSV están en `data/raw/` y usan punto y coma como separador. Los archivos que se generan se guardan en `data/processed/`. Las variables se nombran como en el material, porque cada lección usa las de la anterior.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')
SALIDA.mkdir(exist_ok=True)

orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

print('pandas', pd.__version__)
for nombre, df in [('orders', orders), ('order_prods', order_prods), ('products', products),
                   ('aisles', aisles), ('departments', departments)]:
    print(nombre, df.shape)

pandas 3.0.5
orders (478967, 6)
order_prods (4545007, 4)
products (49694, 4)
aisles (134, 2)
departments (21, 2)


Las cinco tablas tienen los tamaños esperados. Problemas que dejó la exploración inicial y que se resuelven en la Lección 2: 15 pedidos duplicados, nulos en tres columnas, columnas enteras leídas como decimales y el valor centinela `missing`.

---
# Lección 2: Limpieza y estandarización de datos crudos

Para cada problema se decide si eliminar, corregir o conservar, según lo que significa el dato. El orden es: duplicados, tipos, nulos, texto, duplicados implícitos y verificación.

## 2.1 Ejemplos de la explicación

### Duplicados exactos: detectar con `duplicated` y eliminar con `drop_duplicates`

In [2]:
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


Son 30 filas: 15 pedidos y sus 15 copias, todas del miércoles (`order_dow = 3`) a las 2 a. m. Que ocurran en la misma franja apunta a un fallo del sistema, así que es seguro borrarlas.

In [3]:
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)

(478952, 6) True


### Tipos de dato con `astype`

In [4]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Antes: 23.0 MB
Después: 7.2 MB


`Int16` con mayúscula es un tipo entero que admite nulos. Con tipos más pequeños, `orders` pasa de 23,0 MB a 7,2 MB.

### Nulos: comprobar la causa antes de decidir

In [5]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())

True


Cada nulo es un primer pedido, así que no hay un pedido anterior. **Decisión: conservar el nulo.** Rellenarlo con 0 inventaría que el cliente volvió el mismo día.

In [6]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


Los 1.258 productos sin nombre están todos en el pasillo 100 y el departamento 21 (`missing`). **Decisión: corregir con `Unknown`**, porque el producto existe y no se puede borrar.

| Columna | Causa | Decisión |
|---|---|---|
| `days_since_prior_order` | Primer pedido del cliente | Conservar el nulo |
| `product_name` | Producto sin datos de catálogo | Rellenar con `Unknown` |
| `add_to_cart_order` | Se investiga en el reto | Conservar el nulo |

### Texto con los métodos `.str`

In [7]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())

Espacios dobles: 403


Espacio invisible: 16
0 0


### Duplicados implícitos

In [8]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))

141
268 de 280


Hay 141 nombres repetidos después de estandarizar, pero tienen `product_id` distintos y casi todos aparecen en pedidos. **Decisión: conservar y documentar.** En los análisis se agrupa por `product_id`, no por nombre.

### Limpieza reutilizable con funciones y verificación con `assert`

In [9]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)

(478952, 6)


In [10]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'
print('Limpieza verificada')

Limpieza verificada


## 2.2 Laboratorio guiado 2: pipeline de limpieza con informe de cambios

**Encargo:** construir una secuencia de pasos de limpieza que parta de los archivos crudos, se ejecute con una sola instrucción y deje un informe de lo que cambió en cada paso.

### Paso 1: recargar los datos crudos

In [11]:
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(478967, 6) (49694, 4) (4545007, 4)


### Paso 2: una función que anota el estado de la tabla

In [12]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

### Paso 3: un paso de limpieza por función

In [13]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

### Paso 4: encadenar los pasos con `pipe`

In [14]:
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

### Paso 5: el informe de cambios

In [15]:
informe = pd.DataFrame(registro)
informe.to_csv(SALIDA / 'cleaning_report.csv', index=False)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,3.1
4,products,1 nombres,49694,0,3.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


**Interpretación del informe:**

- Quitar duplicados eliminó 15 filas de `orders` y 2 nulos: dos de los pedidos repetidos eran primeros pedidos.
- Corregir los tipos no cambió filas ni nulos, pero redujo la memoria de `orders` de 23,0 a 7,2 MB y la de `order_prods` de 145,4 a 54,5 MB.
- Estandarizar los nombres dejó `products` sin nulos.
- Los 836 nulos de `order_prods` siguen ahí a propósito: se decidió conservarlos.

El informe se guardó en `data/processed/cleaning_report.csv`.

### Paso 6: validar el resultado

In [16]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))

Validación superada
True True


El pipeline pasa todas las reglas de calidad y produce exactamente las mismas tablas que la limpieza manual (`True True`).

**Conclusiones del laboratorio 2:**

- Un pipeline es una cadena de funciones pequeñas que reciben y devuelven un DataFrame, unidas con `pipe`.
- La función `anotar` genera el informe de cambios sin esfuerzo extra y responde con cifras a "¿qué le hiciste a los datos?".
- Validar con `assert` convierte las reglas de calidad en una prueba que se ejecuta cada vez.

## 2.3 Reto 2

**Ejercicio 1.** Comprueba si `order_prods` tiene filas duplicadas exactas, si se repite algún par `order_id` + `product_id` y si `reordered` solo contiene 0 y 1.

In [17]:
print('Filas duplicadas exactas:', order_prods.duplicated().sum())
print('Pares order_id + product_id repetidos:', order_prods.duplicated(['order_id', 'product_id']).sum())
print('Valores de reordered:', sorted(order_prods['reordered'].unique().tolist()))

Filas duplicadas exactas: 0


Pares order_id + product_id repetidos: 0
Valores de reordered: [0, 1]


**Respuesta:** `order_prods` no tiene filas duplicadas exactas (0) ni pares `order_id` + `product_id` repetidos (0), y `reordered` solo contiene 0 y 1. No necesita limpieza por duplicados.

**Ejercicio 2.** Comprueba si `aisles` y `departments` tienen nombres duplicados o con espacios al inicio o al final. ¿Necesitan limpieza de texto?

In [18]:
for nombre, df, col in [('aisles', aisles, 'aisle'), ('departments', departments, 'department')]:
    duplicados_nombre = df[col].duplicated().sum()
    con_espacios = (df[col] != df[col].str.strip()).sum()
    print(f'{nombre}: {duplicados_nombre} nombres duplicados, {con_espacios} con espacios al inicio o al final')

aisles: 0 nombres duplicados, 0 con espacios al inicio o al final
departments: 0 nombres duplicados, 0 con espacios al inicio o al final


**Respuesta:** Ninguna de las dos tablas tiene nombres duplicados ni espacios sobrantes al inicio o al final, así que no necesitan limpieza de texto. Su único detalle es el valor centinela `missing` (pasillo 100 y departamento 21), que es un problema de significado, no de formato.

**Ejercicio 3.** Investiga los 836 nulos de `add_to_cart_order`: ¿en cuántos pedidos ocurren y cuántos productos tiene, como mínimo, cada uno? ¿Qué los explica?

In [19]:
pedidos_con_nulos = order_prods.loc[order_prods['add_to_cart_order'].isna(), 'order_id'].unique()
tamano_pedidos = (order_prods[order_prods['order_id'].isin(pedidos_con_nulos)]
                  .groupby('order_id').size())

print('Pedidos con nulos:', len(pedidos_con_nulos))
print('Productos mínimos en esos pedidos:', tamano_pedidos.min())
print('Posición máxima registrada en el carrito:', order_prods['add_to_cart_order'].max())

Pedidos con nulos: 70
Productos mínimos en esos pedidos: 65
Posición máxima registrada en el carrito: 64


**Respuesta:** Los 836 nulos están en solo **70 pedidos**, y cada uno tiene **como mínimo 65 productos**. Como la posición máxima registrada es 64, el sistema solo guardó la posición hasta el puesto 64: los artículos agregados después quedaron sin posición. Por eso se conservan los nulos: el producto sí se compró, solo falta su posición.

**Ejercicio 4.** Carga de nuevo `products.csv` en una variable `products_crudo` y cuenta cuántos nombres (no nulos) cambiaron con la estandarización.

In [20]:
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
tenia_nombre = products_crudo['product_name'].notna()
cambiaron = (products_crudo.loc[tenia_nombre, 'product_name'] != products.loc[tenia_nombre, 'product_name']).sum()
print('Nombres no nulos:', tenia_nombre.sum())
print('Nombres que cambiaron:', cambiaron)

Nombres no nulos: 48436
Nombres que cambiaron: 6321


**Respuesta:** De los 48.436 nombres no nulos, **6.321** cambiaron con la estandarización (un 13 %): se quitaron espacios dobles e invisibles y se aplicó mayúscula inicial en cada palabra.

**Ejercicio 5.** Crea una Series con el nombre del día de cada pedido y conviértela a una categoría ordenada de domingo a sábado. Compara la memoria del texto con la de la categoría.

In [21]:
dias = {0: 'domingo', 1: 'lunes', 2: 'martes', 3: 'miércoles',
        4: 'jueves', 5: 'viernes', 6: 'sábado'}
dia_texto = orders['order_dow'].map(dias)
dia_categoria = pd.Series(pd.Categorical(dia_texto, categories=list(dias.values()), ordered=True))

print(dia_categoria.dtype)
print('Memoria como texto:    ', round(dia_texto.memory_usage(deep=True) / 1e6, 2), 'MB')
print('Memoria como categoría:', round(dia_categoria.memory_usage(deep=True) / 1e6, 2), 'MB')

category
Memoria como texto:     7.14 MB
Memoria como categoría: 0.48 MB


**Respuesta:** Como texto, la Series ocupa **7,14 MB**; como categoría ordenada, **0,48 MB**, unas 15 veces menos. La categoría guarda los 7 nombres una sola vez y en cada fila solo un código pequeño. Además, al ser ordenada, respeta el orden de domingo a sábado al ordenar o graficar.

**Ejercicio 6.** ¿Cuántas filas de `order_prods` corresponden a productos `Unknown` y en cuántos pedidos aparecen? ¿Los eliminarías? Justifica.

In [22]:
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']
filas_unknown = order_prods[order_prods['product_id'].isin(ids_unknown)]
print('Filas con productos Unknown:', len(filas_unknown))
print('Pedidos en los que aparecen:', filas_unknown['order_id'].nunique())
print('Porcentaje de las filas:', round(len(filas_unknown) / len(order_prods) * 100, 2), '%')

Filas con productos Unknown: 9547
Pedidos en los que aparecen: 8254
Porcentaje de las filas: 0.21 %


**Respuesta:** Hay **9.547 filas** con productos `Unknown` (0,21 % de los artículos), repartidas en **8.254 pedidos**. **No las eliminaría**: son compras reales, y borrarlas cambiaría el tamaño de esos pedidos y las tasas de recompra. Basta con excluirlas cuando el análisis sea por nombre o por categoría de producto.

**Ejercicio 7.** En una copia de `departments`, reemplaza `missing` por un nulo real (`pd.NA`). Explica en una frase por qué no conviene hacerlo en la tabla original antes de agrupar.

In [23]:
departments_na = departments.copy()
departments_na['department'] = departments_na['department'].replace('missing', pd.NA)
print(departments_na.tail(3))
print('Nulos:', departments_na['department'].isna().sum())

    department_id department
18             19     snacks
19             20       deli
20             21        NaN
Nulos: 1


**Respuesta:** En la copia, `missing` pasa a ser un nulo real. No conviene hacerlo en la tabla original antes de agrupar, porque `groupby` descarta por defecto las claves nulas y los productos de ese departamento desaparecerían de los totales sin aviso.

## 2.4 Resultados de la limpieza

| Problema | Evidencia | Decisión | Resultado |
|---|---|---|---|
| Pedidos duplicados en `orders` | 15 filas idénticas, todas del miércoles a las 2 a. m. | Eliminar | 478.952 pedidos y `order_id` único |
| Tipos de dato demasiado grandes y enteros leídos como decimales | `float64` en columnas enteras | Corregir con `astype` y tipos `Int16` que admiten nulos | `orders` de 23,0 a 7,2 MB y `order_prods` de 145,4 a 54,5 MB |
| Nulos en `days_since_prior_order` | 28.817 después de quitar duplicados, todos primeros pedidos | Conservar | La media de espera no se distorsiona |
| Productos sin nombre | 1.258, todos en pasillo y departamento `missing` | Rellenar con `Unknown` | 0 nulos en `product_name` |
| Texto sin estandarizar | 403 nombres con espacios dobles y 16 con espacios invisibles | Corregir con métodos `.str` | 0 y 0; 6.321 nombres cambiaron |
| Duplicados implícitos de nombre | 141 nombres repetidos con `product_id` distinto | Conservar y documentar | Se agrupa por `product_id` |
| Nulos en `add_to_cart_order` | 836 en 70 pedidos de 65 productos o más | Conservar | Posiciones después del puesto 64 sin registrar |

El informe de cambios del pipeline quedó en `data/processed/cleaning_report.csv`. Después de la limpieza, las tablas pasan todas las reglas de validación.

---
# Lección 3: Filtrado, ordenamiento y agrupación de datos

Filtrar elige las filas que importan, ordenar las jerarquiza y agrupar las resume. Antes de escribir código, cada pregunta se traduce a la frase "métrica por grupo": "¿a qué hora se compra más?" se convierte en "número de pedidos por hora".

## 3.1 Ejemplos de la explicación

### Filtrado con máscaras booleanas

In [24]:
es_primer_pedido = orders['order_number'] == 1
print(es_primer_pedido.head(3))
primeros = orders[es_primer_pedido]
print(len(primeros))

0    False
1    False
2    False
Name: order_number, dtype: bool
28817


In [25]:
es_fin_de_semana = orders['order_dow'].isin([0, 6])
es_manana = orders['order_hour_of_day'].between(6, 11)
print(len(orders[es_fin_de_semana & es_manana]))

45599


Hay 45.599 pedidos de fin de semana hechos entre las 6 y las 11 a. m. Con `&` se exige que se cumplan las dos condiciones.

### Filtrado con `query`

In [26]:
nocturnos_30 = orders.query('order_hour_of_day >= 22 and days_since_prior_order == 30')
print(len(nocturnos_30))

1449


### Ordenamiento con `sort_values` y `nlargest`

In [27]:
print(orders.nlargest(3, 'order_number')[['user_id', 'order_number']])
historial = orders.sort_values(['user_id', 'order_number'])
print(historial[['user_id', 'order_number']].head(3))

      user_id  order_number
2482     8220           100
4552   164906           100
6402   172294           100


        user_id  order_number
237012        2             1
317136        2             4
337239        4             2


### Agrupación con `groupby`

In [28]:
pedidos_por_hora = orders.groupby('order_hour_of_day')['order_id'].count()
print(pedidos_por_hora.idxmax(), pedidos_por_hora.max())

10 40578


La hora pico es las 10 a. m., con 40.578 pedidos.

In [29]:
resumen_dia = (orders
    .groupby('order_dow')
    .agg(pedidos=('order_id', 'count'),
         clientes=('user_id', 'nunique'),
         espera_media=('days_since_prior_order', 'mean'))
    .round(1))
resumen_dia

,pedidos,clientes,espera_media
order_dow,,,
0,84090,57661,11.8
1,82185,56479,11.3
2,65833,48587,11.2
3,60897,45331,10.7
4,59810,44281,10.5
5,63488,46127,10.5
6,62649,45604,11.4


El domingo (0) y el lunes (1) tienen más pedidos y clientes. La espera media entre pedidos es parecida todos los días (de 10,5 a 11,8 días).

### Tabla dinámica con `pivot_table`

In [30]:
matriz = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                            values='order_id', aggfunc='count')
print(matriz.shape)
matriz.loc[[8, 10, 14, 20]]

(24, 7)


order_dow,0,1,2,3,4,5,6
order_hour_of_day,,,,,,,
8,3908,4751,3485,3125,3136,3373,3246
10,6978,7881,5489,5026,4938,5347,4919
14,7533,6545,5344,4774,4795,5265,5375
20,2544,2332,2061,1917,2037,1886,1847


### Valores por fila con `transform`

In [31]:
orders['pedidos_cliente'] = orders.groupby('user_id')['order_id'].transform('count')
frecuentes = orders[orders['pedidos_cliente'] >= 5]
print(len(frecuentes), frecuentes['user_id'].nunique())
orders = orders.drop(columns='pedidos_cliente')

231946 30527


## 3.2 Laboratorio guiado 3: perfil de hábitos de compra

**Encargo:** el área de mercadeo quiere diseñar campañas según qué tan seguido compra cada cliente, y necesita saber a qué hora compra cada tipo de cliente.

### Paso 1: pedidos por cliente

In [32]:
pedidos_por_cliente = orders.groupby('user_id')['order_id'].count()
print(len(pedidos_por_cliente))
pedidos_por_cliente.describe()

157437


count    157437.000000
mean          3.042182
std           2.746842
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          28.000000
Name: order_id, dtype: float64

### Paso 2: segmentar clientes con `pd.cut`

In [33]:
segmento_cliente = pd.cut(pedidos_por_cliente,
                          bins=[0, 1, 4, np.inf],
                          labels=['ocasional', 'regular', 'frecuente'])
print(segmento_cliente.value_counts().sort_index())
print((segmento_cliente.value_counts(normalize=True).sort_index() * 100).round(1))

order_id
ocasional    55357
regular      71553
frecuente    30527
Name: count, dtype: int64
order_id
ocasional    35.2
regular      45.4
frecuente    19.4
Name: proportion, dtype: float64


### Paso 3: llevar el segmento a cada pedido con `map`

In [34]:
orders['segmento'] = orders['user_id'].map(segmento_cliente)
orders[['user_id', 'order_number', 'segmento']].head()

,user_id,order_number,segmento
0,183418,11,regular
1,163593,5,regular
2,39980,4,ocasional
3,82516,56,frecuente
4,196724,2,ocasional


### Paso 4: resumen por segmento

In [35]:
resumen_segmento = (orders
    .groupby('segmento', observed=True)
    .agg(clientes=('user_id', 'nunique'),
         pedidos=('order_id', 'count'),
         espera_mediana=('days_since_prior_order', 'median')))
resumen_segmento

,clientes,pedidos,espera_mediana
segmento,,,
ocasional,55357,55357,16.0
regular,71553,191649,11.0
frecuente,30527,231946,6.0


**Interpretación:** el 19,4 % de los clientes (los frecuentes) hace casi la mitad de los pedidos (231.946 de 478.952) y vuelve cada 6 días (mediana). Los ocasionales esperan 16 días porque su único pedido en estos datos no es necesariamente el primero que hicieron en la tienda.

### Paso 5: franjas horarias y tabla cruzada

In [36]:
orders['franja'] = pd.cut(orders['order_hour_of_day'],
                          bins=[-1, 5, 11, 17, 23],
                          labels=['madrugada', 'mañana', 'tarde', 'noche'])

perfil = (pd.crosstab(orders['segmento'], orders['franja'], normalize='index')
            .mul(100).round(1))
perfil

franja,madrugada,mañana,tarde,noche
segmento,,,,
ocasional,2.1,30.0,49.4,18.5
regular,1.9,31.6,48.4,18.1
frecuente,1.7,35.2,45.9,17.2


**Interpretación:** todos los segmentos compran sobre todo en la tarde, pero los frecuentes compran más en la mañana (35,2 % frente a 30,0 % de los ocasionales). Una campaña de fidelización debería llegar por la mañana.

### Paso 6: las combinaciones de día y franja más activas

In [37]:
(orders.groupby(['order_dow', 'franja'], observed=True)
       .size()
       .sort_values(ascending=False)
       .head(3))

order_dow  franja
0          tarde     42340
1          tarde     37432
2          tarde     30751
dtype: int64

Las combinaciones más activas son domingo en la tarde, lunes en la tarde y martes en la tarde.

### Paso 7: dejar `orders` como estaba

In [38]:
orders = orders.drop(columns=['segmento', 'franja'])
print(orders.shape)

(478952, 6)


**Conclusiones del laboratorio 3:**

- Una pregunta de negocio compleja se resuelve en pasos: agrupar, categorizar, devolver al detalle y cruzar.
- `pd.cut` convierte números en categorías con límites explícitos.
- `map` lleva un resultado calculado por grupo a cada fila del detalle.
- `crosstab` con `normalize` permite comparar grupos de distinto tamaño.

## 3.3 Reto 3

**Ejercicio 1.** ¿Cuáles son las tres horas con más pedidos los domingos y cuántos pedidos tiene cada una?<br>*Métrica por grupo: número de pedidos por hora, solo los domingos.*

In [39]:
domingos = orders[orders['order_dow'] == 0]
domingos.groupby('order_hour_of_day').size().nlargest(3)

order_hour_of_day
15    7585
14    7533
13    7483
dtype: int64

**Respuesta:** Los domingos, las tres horas con más pedidos son las **15 h (7.585 pedidos)**, las **14 h (7.533)** y las **13 h (7.483)**. El domingo el pico se corre hacia la tarde, a diferencia de la semana completa, cuyo pico es a las 10 a. m.

**Ejercicio 2.** ¿A qué hora es mayor la espera media entre pedidos (`days_since_prior_order`) y a qué hora es menor?<br>*Métrica por grupo: espera media por hora.*

In [40]:
espera_por_hora = orders.groupby('order_hour_of_day')['days_since_prior_order'].mean().round(2)
print('Mayor espera:', espera_por_hora.idxmax(), 'h ->', espera_por_hora.max(), 'días')
print('Menor espera:', espera_por_hora.idxmin(), 'h ->', espera_por_hora.min(), 'días')

Mayor espera: 0 h -> 12.52 días
Menor espera: 22 h -> 10.62 días


**Respuesta:** La espera media es **mayor a las 0 h (12,52 días)** y **menor a las 22 h (10,62 días)**. La diferencia es de casi dos días: quienes compran a medianoche tardan un poco más en volver.

**Ejercicio 3.** ¿Cuántos clientes tienen 10 pedidos o más?<br>*Métrica por grupo: pedidos por cliente; condición: 10 o más.*

In [41]:
pedidos_cliente = orders.groupby('user_id')['order_id'].count()
print((pedidos_cliente >= 10).sum())

6327


**Respuesta:** **6.327 clientes** tienen 10 pedidos o más en estos datos.

**Ejercicio 4.** ¿Cuántos pedidos se hicieron exactamente 7 días después del anterior y qué porcentaje del total representan?<br>*Condición: `days_since_prior_order == 7`.*

In [42]:
siete_dias = (orders['days_since_prior_order'] == 7).sum()
print('Pedidos a los 7 días:', siete_dias)
print('Porcentaje del total de pedidos:', round(siete_dias / len(orders) * 100, 2), '%')
print('Esperas más comunes:')
print(orders['days_since_prior_order'].value_counts().head(3))

Pedidos a los 7 días: 44577
Porcentaje del total de pedidos: 9.31 %
Esperas más comunes:
days_since_prior_order
30    51337
7     44577
6     33930
Name: count, dtype: Int64


**Respuesta:** **44.577 pedidos** se hicieron exactamente 7 días después del anterior, el **9,31 %** del total de pedidos (478.952). Si se cuentan solo los pedidos que tienen el dato, sin los primeros pedidos, el porcentaje sube a 9,9 %. Es el valor de espera más común después de 30: muchos clientes compran cada semana.

**Ejercicio 5.** Usando `order_prods`: ¿cuántos pedidos tienen 20 productos o más? ¿Cuál es el pedido más grande y cuántos productos tiene?<br>*Métrica por grupo: productos por pedido.*

In [43]:
productos_por_pedido = order_prods.groupby('order_id').size()
print('Pedidos con 20 productos o más:', (productos_por_pedido >= 20).sum())
print('Pedido más grande:', productos_por_pedido.idxmax(), 'con', productos_por_pedido.max(), 'productos')

Pedidos con 20 productos o más: 49420
Pedido más grande: 61355 con 127 productos


**Respuesta:** **49.420 pedidos** tienen 20 productos o más. El pedido más grande es el **61355**, con **127 productos**.

**Ejercicio 6.** Usando `order_prods`: calcula la tasa de recompra (media de `reordered`) para las posiciones 1 a 5 del carrito. ¿Qué patrón ves?<br>*Métrica por grupo: media de `reordered` por posición, solo posiciones 1 a 5.*

In [44]:
primeras_posiciones = order_prods[order_prods['add_to_cart_order'].between(1, 5)]
primeras_posiciones.groupby('add_to_cart_order')['reordered'].mean().round(3)

add_to_cart_order
1    0.679
2    0.677
3    0.659
4    0.638
5    0.617
Name: reordered, dtype: float64

**Respuesta:** La tasa de recompra baja a medida que avanza la posición en el carrito: **0,679** en la posición 1, 0,677 en la 2, 0,659 en la 3, 0,638 en la 4 y **0,617** en la 5. **Patrón:** los clientes agregan primero los productos que ya conocen y compran siempre, y después los nuevos.

**Ejercicio 7.** Compara la mediana de `days_since_prior_order` de los pedidos de fin de semana (domingo y sábado) con la del resto de la semana.<br>*Métrica por grupo: mediana de espera por tipo de día.*

In [45]:
fin_de_semana = orders['order_dow'].isin([0, 6]).rename('fin_de_semana')
orders.groupby(fin_de_semana)['days_since_prior_order'].median()

fin_de_semana
False    7.0
True     8.0
Name: days_since_prior_order, dtype: Float64

**Respuesta:** La mediana de espera es de **8 días** en los pedidos de fin de semana (domingo y sábado) y de **7 días** el resto de la semana. Quienes compran el fin de semana tardan, en la mitad de los casos, un día más en volver.

## 3.4 Interpretación de los resultados

- **Cuándo se compra:** la hora pico es las 10 a. m. (40.578 pedidos), el domingo y el lunes son los días más activos y la tarde es la franja con más pedidos en todos los segmentos. Los domingos el pico se corre a las 3 p. m.
- **Quién compra:** el 19,4 % de los clientes (los frecuentes, con 5 pedidos o más) hace casi la mitad de los pedidos y vuelve cada 6 días (mediana); 6.327 clientes tienen 10 pedidos o más.
- **Con qué frecuencia:** 7 días es una espera muy común (9,3 % de los pedidos), lo que muestra un hábito de compra semanal. El fin de semana la mediana de espera sube a 8 días.
- **Qué tan grande es el pedido:** 49.420 pedidos tienen 20 productos o más; el más grande tiene 127.
- **Recompra:** los primeros productos del carrito son los que más se repiten (tasa de 0,679 en la posición 1).

Para mercadeo, la mejor ventana es la mañana para los clientes frecuentes y la tarde del domingo y el lunes para el resto.

---
# Lección 4: Combinación y estructuración de múltiples datasets

`merge` une columnas de dos tablas emparejando sus claves y `concat` apila filas de tablas con las mismas columnas. Un `merge` mal hecho no da error: multiplica o pierde filas en silencio. Por eso cada unión se comprueba.

## 4.1 Ejemplos de la explicación

### Tipos de unión con el parámetro `how`

In [46]:
pedidos = pd.DataFrame({'order_id': [1, 2, 3], 'user_id': [10, 20, 30]})
detalle = pd.DataFrame({'order_id': [1, 1, 2, 4], 'product_id': [100, 200, 100, 300]})

for tipo in ['inner', 'left', 'outer']:
    resultado = pedidos.merge(detalle, on='order_id', how=tipo)
    print(tipo, len(resultado))
    print(resultado, '\n')

inner 3
   order_id  user_id  product_id
0         1       10         100
1         1       10         200
2         2       20         100 

left 4
   order_id  user_id  product_id
0         1       10       100.0
1         1       10       200.0
2         2       20       100.0
3         3       30         NaN 

outer 5
   order_id  user_id  product_id
0         1     10.0       100.0
1         1     10.0       200.0
2         2     20.0       100.0
3         3     30.0         NaN
4         4      NaN       300.0 



`inner` deja solo las claves que están en las dos tablas (3 filas), `left` conserva todos los pedidos (4) y `outer` conserva todo (5). El pedido 1 aparece dos veces porque tiene dos productos.

### Validación de la relación con `validate`

In [47]:
catalogo_malo = pd.concat([products, products.head(1)])

try:
    order_prods.merge(catalogo_malo, on='product_id', validate='m:1')
except pd.errors.MergeError as error:
    print('Detectado:', str(error).splitlines()[0])

Detectado: Merge keys are not unique in right dataset; not a many-to-one merge


### Auditoría de la unión con `indicator`

In [48]:
auditoria = orders.merge(order_prods[['order_id']].drop_duplicates(),
                         on='order_id', how='left', indicator=True)
auditoria['_merge'].value_counts()

_merge
both          450046
left_only      28906
right_only         0
Name: count, dtype: int64

Hay 28.906 pedidos (6 %) sin productos registrados. Con un `inner` habrían desaparecido sin aviso.

### Construcción de la tabla maestra

In [49]:
filas_inicio = len(order_prods)

maestra = order_prods.merge(products, on='product_id', how='left', validate='m:1')
print('+ products   ', maestra.shape)
maestra = maestra.merge(aisles, on='aisle_id', how='left', validate='m:1')
print('+ aisles     ', maestra.shape)
maestra = maestra.merge(departments, on='department_id', how='left', validate='m:1')
print('+ departments', maestra.shape)
maestra = maestra.merge(orders, on='order_id', how='left', validate='m:1')
print('+ orders     ', maestra.shape)

assert len(maestra) == filas_inicio

+ products    (4545007, 7)


+ aisles      (4545007, 8)


+ departments (4545007, 9)


+ orders      (4545007, 14)


Cada unión añade columnas sin cambiar el número de filas: la tabla maestra conserva los 4.545.007 artículos y termina con 14 columnas. El `assert` confirma que no se crearon ni se perdieron filas.

### Preguntas sobre la tabla maestra

In [50]:
top = (maestra.groupby(['product_id', 'product_name']).size()
              .nlargest(5).reset_index(name='veces'))
print(top)

por_depto = (maestra.groupby('department')
                    .agg(articulos=('product_id', 'size'),
                         tasa_recompra=('reordered', 'mean'))
                    .sort_values('articulos', ascending=False))
por_depto.head(3)

   product_id            product_name  veces
0       24852                  Banana  66050
1       13176  Bag Of Organic Bananas  53297
2       21137    Organic Strawberries  37039
3       21903    Organic Baby Spinach  33971
4       47209    Organic Hass Avocado  29773


,articulos,tasa_recompra
department,,
produce,1328302,0.651140
dairy eggs,759842,0.670597
snacks,404850,0.574442


### Apilado con `concat`

In [51]:
categorias = pd.concat([
    aisles.rename(columns={'aisle_id': 'id', 'aisle': 'nombre'}).assign(tipo='pasillo'),
    departments.rename(columns={'department_id': 'id', 'department': 'nombre'}).assign(tipo='departamento'),
], ignore_index=True)
print(categorias.shape)

(155, 3)


### Formato ancho y largo con `pivot_table` y `melt`

In [52]:
ancho = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                           values='order_id', aggfunc='count')
largo = (ancho.reset_index()
              .melt(id_vars='order_hour_of_day', var_name='order_dow', value_name='pedidos'))
print(ancho.shape, largo.shape)

(24, 7) (168, 3)


### Guardado del resultado

In [53]:
maestra.to_parquet(SALIDA / 'instacart_master.parquet', index=False)
por_depto.to_csv(SALIDA / 'department_summary.csv')
print(sorted(p.name for p in SALIDA.iterdir()))

['.gitkeep', 'cleaning_report.csv', 'department_summary.csv', 'instacart_master.parquet']


La tabla maestra se guarda en Parquet, que conserva los tipos y ocupa menos (requiere `pyarrow`), y el resumen por departamento en CSV para abrirlo en Excel.

## 4.2 Laboratorio guiado 4: tabla de análisis por producto

**Encargo:** el equipo de compras quiere una tabla con una fila por producto que diga cuántas veces se pidió, cuántos clientes lo compran, qué tan seguido se recompra y en qué posición suele ir en el carrito, para saber cuál es el producto más fiel de cada departamento.

### Paso 1: agregar la tabla maestra por producto

In [54]:
por_producto = (maestra
    .groupby('product_id')
    .agg(veces=('order_id', 'size'),
         clientes=('user_id', 'nunique'),
         tasa_recompra=('reordered', 'mean'),
         posicion_media=('add_to_cart_order', 'mean'))
    .reset_index())
print(por_producto.shape)
por_producto.head(3)

(45573, 5)


,product_id,veces,clientes,tasa_recompra,posicion_media
0,1,280,206,0.564286,5.642857
1,2,11,11,0.000000,8.727273
2,3,42,27,0.738095,6.952381


### Paso 2: recuperar nombre y departamento con `merge`

In [55]:
ficha_productos = (por_producto
    .merge(products, on='product_id', how='left', validate='1:1')
    .merge(departments, on='department_id', how='left', validate='m:1')
    [['product_id', 'product_name', 'department', 'veces', 'clientes',
      'tasa_recompra', 'posicion_media']])
print(ficha_productos.shape)
ficha_productos.loc[ficha_productos['veces'].idxmax()]

(45573, 7)


product_id           24852
product_name        Banana
department         produce
veces                66050
clientes             33274
tasa_recompra     0.844254
posicion_media    4.874815
Name: 22808, dtype: object

### Paso 3: filtrar los productos con suficientes datos

In [56]:
populares = ficha_productos[ficha_productos['veces'] >= 1000].copy()
print(len(populares))

739


Una tasa de recompra calculada con pocos pedidos no es confiable, así que se dejan solo los productos pedidos al menos 1.000 veces.

### Paso 4: clasificar dentro de cada departamento con `rank`

In [57]:
populares['puesto'] = (populares
    .groupby('department')['tasa_recompra']
    .rank(ascending=False, method='first'))

lideres = (populares[populares['puesto'] == 1]
           .sort_values('tasa_recompra', ascending=False))
print(len(lideres))
lideres[['department', 'product_name', 'veces', 'tasa_recompra']].head(5)

18


,department,product_name,veces,tasa_recompra
41704,dairy eggs,Whole Organic Omega 3 Milk,1286,0.863142
22808,produce,Banana,66050,0.844254
29030,snacks,Extra Fancy Unsalted Mixed Nuts,1436,0.822423
42302,beverages,Zero Calorie Cola,1256,0.818471
3570,bakery,Organic Honey Sweet Whole Wheat Bread,1607,0.799004


**Interpretación:** el producto más pedido no siempre es el más fiel. En lácteos, la leche Omega 3 tiene pocos pedidos (1.286), pero quien la compra casi siempre la repite.

### Paso 5: la misma respuesta con `sort_values` y `head`

In [58]:
lideres_2 = (populares
    .sort_values('tasa_recompra', ascending=False)
    .groupby('department')
    .head(1))
print(len(lideres_2), set(lideres_2['product_id']) == set(lideres['product_id']))

18 True


### Paso 6: guardar la ficha de productos

In [59]:
ficha_productos.to_parquet(SALIDA / 'product_profile.parquet', index=False)
lideres.to_csv(SALIDA / 'department_leaders.csv', index=False)
print(sorted(p.name for p in SALIDA.iterdir()))

['.gitkeep', 'cleaning_report.csv', 'department_leaders.csv', 'department_summary.csv', 'instacart_master.parquet', 'product_profile.parquet']


**Conclusiones del laboratorio 4:**

- Agregar con `groupby` cambia el nivel de la tabla: de artículos a productos.
- Después de agregar, un `merge` con `validate='1:1'` recupera las columnas descriptivas.
- Filtrar por un mínimo de casos evita conclusiones basadas en pocos datos.
- `groupby().rank()` y `sort_values().groupby().head()` resuelven "el mejor de cada grupo".

## 4.3 Reto 4

Se resuelve usando la tabla maestra.

**Ejercicio 1.** Entre los pasillos con al menos 10.000 artículos pedidos, ¿cuáles son los tres con mayor tasa de recompra? ¿Qué pasillo tiene la tasa más baja de todos?

In [60]:
por_pasillo = maestra.groupby('aisle').agg(articulos=('product_id', 'size'),
                                           tasa_recompra=('reordered', 'mean'))
grandes = por_pasillo[por_pasillo['articulos'] >= 10000]

print('Pasillos con al menos 10.000 artículos:', len(grandes))
print(grandes.nlargest(3, 'tasa_recompra').round(3))
print('\nTasa más baja entre esos pasillos:')
print(grandes.nsmallest(1, 'tasa_recompra').round(3))
print('\nTasa más baja entre todos los pasillos:')
print(por_pasillo.nsmallest(1, 'tasa_recompra').round(3))

Pasillos con al menos 10.000 artículos: 77
                               articulos  tasa_recompra
aisle                                                  
milk                              125522          0.781
water seltzer sparkling water     117133          0.731
fresh fruits                      510101          0.718

Tasa más baja entre esos pasillos:
                   articulos  tasa_recompra
aisle                                      
spices seasonings      29454          0.152

Tasa más baja entre todos los pasillos:
                   articulos  tasa_recompra
aisle                                      
spices seasonings      29454          0.152


**Respuesta:** De los 77 pasillos con al menos 10.000 artículos, los tres con mayor tasa de recompra son **milk (0,781)**, **water seltzer sparkling water (0,731)** y **fresh fruits (0,718)**. El pasillo con la tasa más baja de todos es **spices seasonings (0,152)**: las especias se compran una vez y duran mucho.

**Ejercicio 2.** Compara la participación de cada departamento en los pedidos de madrugada (0 a 5 h) con su participación total. ¿Qué departamento entra en el top 3 de madrugada pero no en el total?

In [61]:
madrugada = maestra[maestra['order_hour_of_day'].between(0, 5)]
participacion = pd.DataFrame({
    'madrugada_%': madrugada['department'].value_counts(normalize=True).mul(100).round(2),
    'total_%': maestra['department'].value_counts(normalize=True).mul(100).round(2),
}).sort_values('madrugada_%', ascending=False)

print(participacion.head(5))
top3_madrugada = set(participacion['madrugada_%'].nlargest(3).index)
top3_total = set(participacion['total_%'].nlargest(3).index)
print('\nEn el top 3 de madrugada pero no en el total:', top3_madrugada - top3_total)

            madrugada_%  total_%
department                      
produce           30.21    29.23
dairy eggs        15.87    16.72
beverages          8.49     8.26
snacks             7.89     8.91
frozen             7.65     6.91

En el top 3 de madrugada pero no en el total: {'beverages'}


**Respuesta:** En la madrugada (0 a 5 h), el top 3 es produce (30,21 %), dairy eggs (15,87 %) y **beverages (8,49 %)**. En el total, el tercer puesto es de snacks (8,91 %). **Beverages entra al top 3 de madrugada pero no al total**, mientras que snacks baja en la madrugada. Frozen también sube en la madrugada (7,65 % frente a 6,91 %).

**Ejercicio 3.** ¿Cuáles son los tres productos más pedidos del departamento snacks?

In [62]:
snacks = maestra[maestra['department'] == 'snacks']
snacks.groupby(['product_id', 'product_name']).size().nlargest(3)

product_id  product_name                        
13870       Lightly Salted Baked Snap Pea Crisps    3462
34448       Original Veggie Straws                  2331
31040       Sea Salt Pita Chips                     2275
dtype: int64

**Respuesta:** Los tres productos más pedidos de snacks son **Lightly Salted Baked Snap Pea Crisps (3.462)**, **Original Veggie Straws (2.331)** y **Sea Salt Pita Chips (2.275)**.

**Ejercicio 4.** Construye una tabla con una fila por cliente y tres columnas: pedidos distintos, artículos y tasa de recompra. ¿Cuántos clientes tiene, cuál es el promedio de artículos por cliente y qué cliente compró más artículos?

In [63]:
por_cliente = maestra.groupby('user_id').agg(pedidos=('order_id', 'nunique'),
                                            articulos=('product_id', 'size'),
                                            tasa_recompra=('reordered', 'mean'))
print('Clientes:', len(por_cliente))
print('Promedio de artículos por cliente:', round(por_cliente['articulos'].mean(), 2))
print('Cliente con más artículos:')
por_cliente.nlargest(1, 'articulos')

Clientes: 149626
Promedio de artículos por cliente: 30.38
Cliente con más artículos:


,pedidos,articulos,tasa_recompra
user_id,,,
103634,18,580,0.822414


**Respuesta:** La tabla tiene **149.626 clientes**, menos que los 157.437 de `orders`, porque los clientes cuyos pedidos no tienen productos registrados no aparecen en la tabla maestra. En promedio, cada cliente compró **30,38 artículos**. El cliente que más compró es el **103634**, con **580 artículos** en 18 pedidos y una tasa de recompra de 0,82.

**Ejercicio 5.** Une `orders` con los pedidos de `order_prods` usando `how='inner'`. ¿Cuántas filas obtienes y por qué son menos que con `how='left'`?

In [64]:
pedidos_con_productos = order_prods[['order_id']].drop_duplicates()
union_inner = orders.merge(pedidos_con_productos, on='order_id', how='inner')
union_left = orders.merge(pedidos_con_productos, on='order_id', how='left')
print('inner:', len(union_inner))
print('left: ', len(union_left))
print('Diferencia:', len(union_left) - len(union_inner))

inner: 450046
left:  478952
Diferencia: 28906


**Respuesta:** Con `how='inner'` se obtienen **450.046 filas** y con `how='left'`, **478.952**. La diferencia (**28.906**) son los pedidos que existen en `orders` pero no tienen productos registrados en `order_prods`: `inner` los elimina sin aviso y `left` los conserva.

**Ejercicio 6.** Con `concat`, arma una sola tabla con los tres productos más pedidos de produce y los tres de dairy eggs, con una columna que indique el departamento.

In [65]:
def top3(departamento):
    return (maestra[maestra['department'] == departamento]
            .groupby('product_name').size()
            .nlargest(3)
            .reset_index(name='veces')
            .assign(departamento=departamento))

top_produce_lacteos = pd.concat([top3('produce'), top3('dairy eggs')], ignore_index=True)
top_produce_lacteos

,product_name,veces,departamento
0,Banana,66050,produce
1,Bag Of Organic Bananas,53297,produce
2,Organic Strawberries,37039,produce
3,Organic Whole Milk,20953,dairy eggs
4,Organic Half & Half,10752,dairy eggs
5,Half & Half,9588,dairy eggs


**Respuesta:** La tabla tiene 6 filas: Banana (66.050), Bag Of Organic Bananas (53.297) y Organic Strawberries (37.039) de produce, y Organic Whole Milk (20.953), Organic Half & Half (10.752) y Half & Half (9.588) de dairy eggs. La columna `departamento` indica de dónde viene cada producto.

**Ejercicio 7.** Crea una tabla dinámica de artículos por departamento (filas) y día de la semana (columnas). ¿Qué forma tiene y qué día se piden más artículos de alcohol?

In [66]:
articulos_depto_dia = maestra.pivot_table(index='department', columns='order_dow',
                                          values='product_id', aggfunc='count')
print('Forma:', articulos_depto_dia.shape)
print('Artículos de alcohol por día:')
print(articulos_depto_dia.loc['alcohol'])
print('Día con más artículos de alcohol:', articulos_depto_dia.loc['alcohol'].idxmax())
dia_pico = articulos_depto_dia.idxmax(axis=1)
print('Departamentos cuyo día pico no es domingo (0) ni lunes (1):', dia_pico[~dia_pico.isin([0, 1])].to_dict())

Forma: (21, 7)
Artículos de alcohol por día:
order_dow
0    2397
1    3025
2    2868
3    3094
4    3656
5    3814
6    2787
Name: alcohol, dtype: int64
Día con más artículos de alcohol: 5
Departamentos cuyo día pico no es domingo (0) ni lunes (1): {'alcohol': 5}


**Respuesta:** La tabla dinámica tiene forma **(21, 7)**: 21 departamentos y 7 días. Los artículos de alcohol se piden más el **día 5, viernes (3.814 artículos)**, seguido del jueves (3.656). Es el único departamento que no sigue el patrón general de domingo y lunes: se compra para el fin de semana.

## 4.4 Validación de las combinaciones y archivos procesados

**Validación de las combinaciones:**

- Cada `merge` de la tabla maestra usó `how='left'` y `validate='m:1'`, y el `assert` confirmó que se mantuvieron las 4.545.007 filas de `order_prods`: no se duplicaron ni se perdieron artículos.
- La prueba con un catálogo con un producto repetido mostró que `validate` detiene la unión en lugar de duplicar filas en silencio.
- La auditoría con `indicator=True` encontró 28.906 pedidos sin productos (`left_only`) y 0 productos sin pedido (`right_only`).
- En el laboratorio 4, el `merge` con `validate='1:1'` confirmó que la ficha tiene un solo registro por producto, y las dos formas de obtener los líderes por departamento dieron el mismo resultado.

**Archivos guardados en `data/processed/`:**

| Archivo | Contenido |
|---|---|
| `cleaning_report.csv` | Informe de cambios del pipeline de limpieza (Lección 2) |
| `instacart_master.parquet` | Tabla maestra: un artículo por fila con producto, pasillo, departamento y datos del pedido (4.545.007 filas, 14 columnas) |
| `department_summary.csv` | Artículos y tasa de recompra por departamento |
| `product_profile.parquet` | Ficha de 45.573 productos: veces pedido, clientes, tasa de recompra y posición media |
| `department_leaders.csv` | Producto con mayor tasa de recompra de cada departamento (18 departamentos) |

La tabla maestra pesa más de 100 MB, el límite de GitHub, así que no se sube al repositorio: se genera al ejecutar este cuaderno.

---
# Conclusiones

**Respuesta a la pregunta orientadora:** los datos de Instacart se preparan en tres etapas. Primero se **limpian** con un pipeline repetible: se eliminan los 15 pedidos duplicados, se corrigen los tipos, se decide qué hacer con cada nulo según su causa y se estandariza el texto, verificando todo con `assert`. Después se **analizan** con filtros, ordenamientos y agrupaciones para responder preguntas de negocio. Por último se **estructuran** en una tabla maestra validada que une las cinco tablas sin perder ni duplicar filas. Así, los resultados sobre los hábitos de compra son confiables.

**Hallazgos principales:**

1. Los clientes de Instacart compran sobre todo de día: la hora pico es las 10 a. m., la tarde es la franja más activa y el domingo y el lunes son los días con más pedidos. El alcohol es la excepción: se compra más los viernes.
2. Un grupo pequeño de clientes frecuentes (19,4 %) hace casi la mitad de los pedidos y vuelve cada 6 días. Muchos clientes tienen un hábito de compra semanal.
3. La recompra es alta en productos básicos como leche, agua y frutas, y baja en productos que duran, como las especias. Los productos que se repiten se agregan primero al carrito.
4. El 6 % de los pedidos no tiene productos registrados. Por eso es necesario auditar cada unión antes de sacar conclusiones.